# 04 — DistilBERT fine-tuning

Candidato B del TDT §7.2: paradigma neuronal contextual. Resuelve la limitación del modelo vectorial (no capta la fuerza ilocutiva) mediante *embeddings* dependientes del contexto.

**Objetivo cuantitativo (criterio E2 §2):** superar de forma consistente el F1-macro del TF-IDF+SVM (~0.79 en test). Referencia publicada sobre NLBSE'23 con Transformers: ~0.86–0.89 F1 (TDT §8.1).

> **Requiere GPU.** Pensada para Google Colab con runtime **T4**. En CPU el fine-tuning es inviable (≫ 1 h). La celda de bootstrap imprime el `device`: si no es `cuda`, cambia el entorno de ejecución antes de seguir.

## 1. Bootstrap

In [ ]:
REPO_URL = 'https://github.com/elvinsomon/pln-poc.git'

import os, sys, subprocess
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.isdir('/content/pln-poc'):
        subprocess.run(['git', 'clone', REPO_URL, '/content/pln-poc'], check=True)
    os.chdir('/content/pln-poc')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        os.chdir('..')

PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
from pathlib import Path
from src.utils.colab import setup_environment, bootstrap_dataset
from src.utils.config import load_config
from src.data.splits import prepare_splits, load_splits
from src.models.bert import build_model, BertClassifier, detect_device
from src.evaluation.metrics import compute_metrics, save_metrics, plot_confusion

setup_environment(seed=42, project_root=PROJECT_ROOT)
cfg_data = load_config('data.yaml')
cfg = load_config('bert.yaml')
labels = cfg['classes']
bootstrap_dataset(cfg_data)
print('cwd   :', os.getcwd())
print('device:', detect_device())   # debe imprimir 'cuda' en Colab con GPU; si dice 'cpu', cambia el runtime

## 2. Cargar splits

> **Decisión:** los mismos parquets que 02/03 (idéntico hold-out estratificado). Comparación justa: BERT ve exactamente el mismo texto anonimizado/limpiado que el SVM.

In [ ]:
prepare_splits(cfg_data, project_root=PROJECT_ROOT)   # idempotente
splits = load_splits(cfg_data, project_root=PROJECT_ROOT)
for name, df in splits.items():
    print(f'{name:5s} n={len(df):>6,}  dist={df["label"].value_counts().to_dict()}')

## 3. Fine-tuning

> **Decisión:** distilbert-base-uncased, 3 épocas, lr 2e-5, fp16, batch 32, max_length 256 (configurable en `configs/bert.yaml`). `Trainer` de HuggingFace. Tiempo esperado en T4 ≈ 6–8 min.

In [ ]:
model = build_model(cfg)
model.fit(splits['train']['text'], splits['train']['label'])

y_val_pred = model.predict(splits['val']['text'])
val_metrics = compute_metrics(splits['val']['label'], y_val_pred, labels=labels)
print('VAL · accuracy :', round(val_metrics['accuracy'], 4))
print('VAL · f1_macro :', round(val_metrics['f1_macro'], 4))

## 4. Evaluación en test

In [ ]:
y_test_pred = model.predict(splits['test']['text'])
test_metrics = compute_metrics(splits['test']['label'], y_test_pred, labels=labels)
print(test_metrics['report'])
print('TEST · accuracy :', round(test_metrics['accuracy'], 4))
print('TEST · f1_macro :', round(test_metrics['f1_macro'], 4))

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(5, 4))
plot_confusion(test_metrics['confusion_matrix']['matrix'], labels=labels, ax=ax, normalize=True)
ax.set_title('DistilBERT · test (normalizada)')
plt.show()

## 5. Teaser: BERT vs TF-IDF+SVM

> **Lectura esperada:** Δ f1_macro positivo. La comparativa completa (tablas + barras + margen relativo) vive en la notebook 05.

In [ ]:
import json
svm_path = Path(cfg['paths']['metrics']) / 'tfidf_svm.json'
if svm_path.exists():
    svm = json.loads(svm_path.read_text(encoding='utf-8'))['test']
    delta = test_metrics['f1_macro'] - svm['f1_macro']
    print(f"SVM  · f1_macro test: {svm['f1_macro']:.4f}")
    print(f"BERT · f1_macro test: {test_metrics['f1_macro']:.4f}")
    print(f"Δ f1_macro (BERT - SVM): {delta:+.4f}")
else:
    print('No existe tfidf_svm.json — ejecuta antes la notebook 03. Comparativa completa en 05.')

## 6. Persistencia (modelo + métricas + predicciones)

In [ ]:
models_dir = Path(cfg['paths']['models']) / 'distilbert'
model.save(models_dir)

save_metrics({'val': val_metrics, 'test': test_metrics},
             f"{cfg['paths']['metrics']}/bert.json")

# Predicciones por fila -> 05 y 06 no necesitan recargar el modelo en GPU.
preds_df = pd.DataFrame({
    'text':  splits['test']['text'].values,
    'label': splits['test']['label'].values,
    'pred_bert': y_test_pred,
})
proba = model.predict_proba(splits['test']['text'])
for i, c in enumerate(labels):
    preds_df[f'proba_bert_{c}'] = proba[:, i]
preds_path = Path(cfg['paths']['reports']) / 'preds_test_bert.parquet'
preds_path.parent.mkdir(parents=True, exist_ok=True)
preds_df.to_parquet(preds_path, index=False)
print('saved:', models_dir)
print('saved:', preds_path)

## 7. Interpretación

- Esperamos que BERT mejore sobre TF-IDF+SVM sobre todo en los pares confusos por **fuerza ilocutiva** (TDT §2.1): `feature ↔ question` y `bug ↔ question`, donde el vocabulario se solapa pero la intención difiere.
- La matriz de confusión normalizada debe mostrar menos masa fuera de la diagonal que la del SVM (notebook 03).
- Los errores residuales son el punto de partida del análisis lingüístico de la notebook 06.